### Wearable Data Pipeline — MinIO → TimescaleDB

**Flow:**
```
MinIO  →  customers/CUST_xxx/wearables/*.csv
                    ↓
          Parse + clean DataFrame
                    ↓
       wearable_readings  (hypertable — raw data)
                    ↓  auto-aggregated
       wearable_hourly   /  wearable_daily
       wearable_weekly   /  wearable_monthly
```

**Expected CSV columns:**
```
timestamp, customer_id, patient_name,
heart_rate, spo2_pct, steps,
skin_temp_c, hrv_ms, respiratory_rate, activity
```

**Duplicate prevention:** `UNIQUE (time, customer_id)` — safe to re-run anytime.

## 1. Install & Import

In [ ]:
!pip install minio psycopg2-binary python-dotenv pandas --quiet

In [1]:
import os
import io
import psycopg2
import pandas as pd
from minio import Minio
from dotenv import load_dotenv

load_dotenv()


True

In [2]:

def get_ts():
    """Connect to TimescaleDB — port 5433 from docker-compose."""
    return psycopg2.connect(
        host     = "localhost",
        port     = 5433,
        dbname   = os.getenv("TIMESCALE_DB"),
        user     = os.getenv("TIMESCALE_USER"),
        password = os.getenv("TIMESCALE_PASSWORD")
    )

def get_minio():
    return Minio(
        "localhost:9000",
        access_key = os.getenv("MINIO_ROOT_USER"),
        secret_key = os.getenv("MINIO_ROOT_PASSWORD"),
        secure     = False
    )

ts    = get_ts()
minio = get_minio()
print("✅ Connected to TimescaleDB and MinIO")

✅ Connected to TimescaleDB and MinIO


---
#### 2. Create Hypertable
Run **once**. Creates the raw readings table and converts it to a TimescaleDB hypertable.

In [3]:
def create_hypertable(ts):
    """Create the raw wearable readings hypertable."""
    with ts.cursor() as cur:

        cur.execute("""
            CREATE TABLE IF NOT EXISTS wearable_readings (
                time               TIMESTAMPTZ  NOT NULL,
                customer_id        TEXT         NOT NULL,
                patient_name       TEXT,
                heart_rate         SMALLINT,        -- bpm
                spo2_pct           SMALLINT,        -- oxygen %
                steps              INT,             -- steps per interval
                skin_temp_c        REAL,            -- °C
                hrv_ms             REAL,            -- heart rate variability ms
                respiratory_rate   REAL,            -- breaths/min
                activity           TEXT,            -- walking | resting | sleeping
                UNIQUE (time, customer_id)
            );
        """)

        cur.execute("""
            SELECT create_hypertable(
                'wearable_readings', 'time',
                if_not_exists => TRUE
            );
        """)

    ts.commit()
    print("✅ Hypertable wearable_readings ready")


create_hypertable(ts)

✅ Hypertable wearable_readings ready


---
#### 3. Create Continuous Aggregates
Run **once**. Each aggregate is its own transaction — one failure won't affect the others.

In [4]:
# (view_name, bucket_size, time_alias)
AGGREGATES = [
    ("wearable_hourly",  "1 hour",  "hour"),
    ("wearable_daily",   "1 day",   "day"),
    ("wearable_weekly",  "7 days",  "week"),
    ("wearable_monthly", "30 days", "month"),
]

def create_aggregate(ts, view, bucket, alias):
    """Create one continuous aggregate view."""
    with ts.cursor() as cur:
        cur.execute(f"""
            CREATE MATERIALIZED VIEW IF NOT EXISTS {view}
            WITH (timescaledb.continuous) AS
            SELECT
                time_bucket('{bucket}', time)          AS {alias},
                customer_id,
                ROUND(AVG(heart_rate)::numeric, 1)     AS avg_heart_rate,
                ROUND(AVG(spo2_pct)::numeric, 1)       AS avg_spo2,
                SUM(steps)                             AS total_steps,
                ROUND(AVG(skin_temp_c)::numeric, 2)    AS avg_skin_temp,
                ROUND(AVG(hrv_ms)::numeric, 2)         AS avg_hrv,
                ROUND(AVG(respiratory_rate)::numeric,2)AS avg_resp_rate
            FROM wearable_readings
            GROUP BY time_bucket('{bucket}', time), customer_id
            WITH NO DATA;
        """)
    ts.commit()


def create_all_aggregates(ts):
    for view, bucket, alias in AGGREGATES:
        try:
            create_aggregate(ts, view, bucket, alias)
            print(f"  ✅ {view}")
        except Exception as e:
            ts.rollback()
            print(f"  ⚠️  {view} skipped — {e}")


create_all_aggregates(ts)

  ✅ wearable_hourly
  ✅ wearable_daily
  ✅ wearable_weekly
  ✅ wearable_monthly


---
#### 4. Add Auto-Refresh Policies
Run **once**. TimescaleDB will auto-refresh aggregates on schedule — no manual refresh needed.

only need to run add_refresh_policies() once ever. After that it lives inside the database permanently.

```
docker-compose up
       ↓
TimescaleDB container starts
       ↓
Reads stored policies from database
       ↓
Background scheduler resumes automatically
       ↓
Policies run on their configured interval
```

#### Aggregate Refresh Policy Schedule

| View | Refreshes Every | Covers Data From | Start Offset | End Offset |
|---|---|---|---|---|
| `wearable_hourly` | 1 hour | last 1 day | 3 hours ago | 30 min before now |
| `wearable_daily` | 1 day | last 2 days | 3 days ago | 1 hour before now |
| `wearable_weekly` | 7 days | last 7 days | 21 days ago | 1 day before now |
| `wearable_monthly` | 30 days | last 30 days | 65 days ago | 1 day before now |

> **Note:** Policies are stored inside TimescaleDB and resume automatically
> every time Docker starts. You only need to run `add_refresh_policies()` once.
>
> **Start offset** — how far back TimescaleDB looks for changed data on each run.
> **End offset** — safety buffer before NOW to avoid computing incomplete buckets.

#### Refresh Policy Window Requirements

| View | Bucket | Minimum Window Needed | Your New Window |
|---|---|---|---|
| `wearable_hourly` | 1 hour | > 2 hours | 3 hours ✅ |
| `wearable_daily` | 1 day | > 2 days | 3 days ✅ |
| `wearable_weekly` | 7 days | > 14 days | 21 days ✅ |
| `wearable_monthly` | 30 days | > 60 days | 65 days ✅ |

> **Rule:** The refresh window (`start_offset` → `end_offset`) must always
> cover at least 2 complete buckets, otherwise TimescaleDB rejects the policy.

In [ ]:
# # (view_name, start_offset, end_offset, refresh_every)
# POLICIES = [
#     ("wearable_hourly",  "-3 hours",   "30 minutes", "1 hour"),
#     ("wearable_daily",   "-3 days",  "1 hour",     "1 day"),
#     ("wearable_weekly",  "-21 days",  "1 day",      "7 days"),
#     ("wearable_monthly", "-65 days", "1 day",      "30 days"),
# ]

# def add_refresh_policies(ts):
#     for view, start, end, interval in POLICIES:
#         try:
#             with ts.cursor() as cur:
#                 cur.execute(f"""
#                     SELECT add_continuous_aggregate_policy(
#                         '{view}',
#                         start_offset      => INTERVAL '{start}',
#                         end_offset        => INTERVAL '{end}',
#                         schedule_interval => INTERVAL '{interval}',
#                         if_not_exists     => TRUE
#                     );
#                 """)
#             ts.commit()
#             print(f"  ✅ Refresh policy → {view} every {interval}")
#         except Exception as e:
#             ts.rollback()
#             print(f"  ⚠️  {view} policy skipped — {e}")


# add_refresh_policies(ts)

  ⚠️  wearable_hourly policy skipped — policy refresh window too small
DETAIL:  The start and end offsets must cover at least two buckets in the valid time range of type "timestamp with time zone".

  ⚠️  wearable_daily policy skipped — policy refresh window too small
DETAIL:  The start and end offsets must cover at least two buckets in the valid time range of type "timestamp with time zone".

  ⚠️  wearable_weekly policy skipped — policy refresh window too small
DETAIL:  The start and end offsets must cover at least two buckets in the valid time range of type "timestamp with time zone".

  ⚠️  wearable_monthly policy skipped — policy refresh window too small
DETAIL:  The start and end offsets must cover at least two buckets in the valid time range of type "timestamp with time zone".



---
#### 5. Read & Parse CSV from MinIO
Handles flexible column names using an alias map.

In [5]:
BUCKET = "health-data"

# Maps possible CSV column names → standard internal names
# Add more aliases here if your CSVs use different naming
COL_ALIASES = {
    "time":             ["timestamp", "datetime", "date_time", "time"],
    "heart_rate":       ["heart_rate", "heart_rate_bpm","heartrate", "hr", "bpm"],
    "spo2_pct":         ["spo2_pct", "spo2", "oxygen", "o2"],
    "steps":            ["steps", "step_count"],
    "skin_temp_c":      ["skin_temp_c", "skin_temp", "temperature", "temp_c"],
    "hrv_ms":           ["hrv_ms", "hrv", "heart_rate_variability"],
    "respiratory_rate": ["respiratory_rate", "resp_rate", "breathing_rate"],
    "activity":         ["activity", "activity_type", "state"],
    "patient_name":     ["patient_name", "name", "patient"],
}

def read_csv(minio, object_path):
    """Download CSV from MinIO, normalise columns, return clean DataFrame."""
    resp = minio.get_object(BUCKET, object_path)
    df   = pd.read_csv(io.BytesIO(resp.read()))
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # rename columns using alias map
    rename = {}
    for std_name, aliases in COL_ALIASES.items():
        for col in df.columns:
            if col in aliases and col != std_name:
                rename[col] = std_name
                break
    df = df.rename(columns=rename)

    # parse timestamp → UTC
    if "time" not in df.columns:
        raise ValueError(f"No timestamp column found. Columns: {list(df.columns)}")
    df["time"] = pd.to_datetime(df["time"], utc=True)

    return df

---
## 6. Insert Readings into TimescaleDB
`ON CONFLICT DO NOTHING` ensures zero duplicates.

In [6]:
INSERT_SQL = """
    INSERT INTO wearable_readings (
        time, customer_id, patient_name,
        heart_rate, spo2_pct, steps,
        skin_temp_c, hrv_ms, respiratory_rate, activity
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (time, customer_id) DO NOTHING
"""

def insert_readings(ts, df, customer_id):
    """Bulk insert DataFrame rows. Returns count of newly inserted rows."""
    records = [
        (
            row["time"],
            customer_id,
            row.get("patient_name"),
            row.get("heart_rate"),
            row.get("spo2_pct"),
            row.get("steps"),
            row.get("skin_temp_c"),
            row.get("hrv_ms"),
            row.get("respiratory_rate"),
            row.get("activity"),
        )
        for _, row in df.iterrows()
    ]
    with ts.cursor() as cur:
        cur.executemany(INSERT_SQL, records)
        inserted = cur.rowcount
    ts.commit()
    return inserted

---
## 7. Sync All Customers from MinIO

Scans every `customers/*/wearables/*.csv` in MinIO.
- ✅ New customer added to MinIO → picked up automatically
- ✅ New CSV for existing customer → only new rows inserted
- ✅ Re-run anytime — duplicates silently skipped

In [7]:
BUCKET = "health-data"
def sync_wearables(minio, ts):
    """Scan MinIO and load all wearable CSVs into TimescaleDB."""
    objects = minio.list_objects(BUCKET, prefix="customers/", recursive=True)
    csvs    = [
        obj for obj in objects
        if "wearables" in obj.object_name
        and obj.object_name.endswith(".csv")
    ]

    print(f"Found {len(csvs)} wearable CSV file(s)\n")
    total_inserted = 0

    for obj in csvs:
        customer_id = obj.object_name.split("/")[1]
        print(f"👤 {customer_id}")
        print(f"   📄 {obj.object_name}")

        try:
            df       = read_csv(minio, obj.object_name)
            inserted = insert_readings(ts, df, customer_id)
            total_inserted += inserted
            skipped = len(df) - inserted
            print(f"   ✅ {inserted} inserted  |  {skipped} duplicates skipped  |  {len(df)} total in file")
        except Exception as e:
            ts.rollback()
            print(f"   ❌ Error: {e}")

    print(f"\n🎉 Sync complete — {total_inserted} total rows inserted")


# ▶️ Run the sync
sync_wearables(minio, ts)

Found 5 wearable CSV file(s)

👤 CUST_amara_patel_03630F04
   📄 customers/CUST_amara_patel_03630F04/wearables/Wearable_CUST_ama_20260511.csv
   ✅ 720 inserted  |  0 duplicates skipped  |  720 total in file
👤 CUST_carlos_rivera_5A81755B
   📄 customers/CUST_carlos_rivera_5A81755B/wearables/Wearable_CUST_car_20260511.csv
   ✅ 720 inserted  |  0 duplicates skipped  |  720 total in file
👤 CUST_fatima_alsayed_8594AA83
   📄 customers/CUST_fatima_alsayed_8594AA83/wearables/Wearable_CUST_fat_20260511.csv
   ✅ 720 inserted  |  0 duplicates skipped  |  720 total in file
👤 CUST_john_whitfield_C4987FD5
   📄 customers/CUST_john_whitfield_C4987FD5/wearables/Wearable_CUST_joh_20260511.csv
   ✅ 720 inserted  |  0 duplicates skipped  |  720 total in file
👤 CUST_mei_lin_66CBFE0F
   📄 customers/CUST_mei_lin_66CBFE0F/wearables/Wearable_CUST_mei_20260511.csv
   ✅ 720 inserted  |  0 duplicates skipped  |  720 total in file

🎉 Sync complete — 3600 total rows inserted


In [9]:
def force_refresh(ts):
    VIEWS = ["wearable_hourly", "wearable_daily", "wearable_weekly", "wearable_monthly"]
    
    # must run outside transaction block
    ts.autocommit = True

    with ts.cursor() as cur:
        for view in VIEWS:
            try:
                cur.execute(f"""
                    CALL refresh_continuous_aggregate(
                        '{view}', '2020-01-01', '2035-12-31'
                    );
                """)
                print(f"  ✅ Refreshed {view}")
            except Exception as e:
                print(f"  ❌ {view} — {e}")

    # turn autocommit back off for normal operations
    ts.autocommit = False

force_refresh(ts)

  ✅ Refreshed wearable_hourly
  ✅ Refreshed wearable_daily
  ✅ Refreshed wearable_weekly
  ✅ Refreshed wearable_monthly


In [10]:
def check_views(ts):
    VIEWS = ["wearable_readings", "wearable_hourly", "wearable_daily", "wearable_weekly", "wearable_monthly"]
    with ts.cursor() as cur:
        for t in VIEWS:
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            count = cur.fetchone()[0]
            print(f"  {t:<25} {count:>8} rows")

check_views(ts)

  wearable_readings             3600 rows
  wearable_hourly               3600 rows
  wearable_daily                 150 rows
  wearable_weekly                 25 rows
  wearable_monthly                10 rows


---
## 8. Manually Refresh Aggregates
The auto-refresh policies handle this automatically. Use this cell to force a refresh after a bulk load.

In [14]:
# def refresh_aggregates(ts):
#     """Force-refresh all continuous aggregate views."""
#     for view, _, alias in AGGREGATES:
#         try:
#             with ts.cursor() as cur:
#                 cur.execute(f"""
#                     CALL refresh_continuous_aggregate(
#                         '{view}', '2020-01-01', '2035-12-31'
#                     );
#                 """)
#             ts.commit()
#             print(f"  🔄 Refreshed {view}")
#         except Exception as e:
#             ts.rollback()
#             print(f"  ❌ {view} — {e}")


# refresh_aggregates(ts)

---
## 9. Verify — Row Counts

In [13]:
def show_counts(ts):
    tables = [
        "wearable_readings",
        "wearable_hourly",
        "wearable_daily",
        "wearable_weekly",
        "wearable_monthly",
    ]
    print("📊 Row counts:")
    with ts.cursor() as cur:
        for t in tables:
            cur.execute(f"SELECT COUNT(*) FROM {t}")
            print(f"  {t:<25} {cur.fetchone()[0]:>8} rows")


show_counts(ts)

📊 Row counts:
  wearable_readings             3600 rows
  wearable_hourly               3600 rows
  wearable_daily                 150 rows
  wearable_weekly                 25 rows
  wearable_monthly                10 rows


---
## 10. Sample Trend Queries

In [15]:
def run_query(ts, sql, label):
    with ts.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(f"\n📊 {label}")
    print("  " + " | ".join(f"{c:<22}" for c in cols))
    print("  " + "-" * 80)
    for row in rows:
        print("  " + " | ".join(f"{str(v):<22}" for v in row))

In [16]:
# Daily summary for all customers
run_query(ts, """
    SELECT day, customer_id,
           avg_heart_rate, avg_spo2,
           total_steps, avg_skin_temp
    FROM wearable_daily
    ORDER BY day DESC, customer_id
    LIMIT 15
""", "Daily Summary — All Customers")


📊 Daily Summary — All Customers
  day                    | customer_id            | avg_heart_rate         | avg_spo2               | total_steps            | avg_skin_temp         
  --------------------------------------------------------------------------------
  2026-05-10 00:00:00+00:00 | CUST_amara_patel_03630F04 | 88.3                   | 96.2                   | 3545                   | 36.80                 
  2026-05-10 00:00:00+00:00 | CUST_carlos_rivera_5A81755B | 67.5                   | 97.9                   | 6187                   | 36.80                 
  2026-05-10 00:00:00+00:00 | CUST_fatima_alsayed_8594AA83 | 71.7                   | 96.7                   | 1941                   | 36.73                 
  2026-05-10 00:00:00+00:00 | CUST_john_whitfield_C4987FD5 | 82.2                   | 95.1                   | 3744                   | 36.84                 
  2026-05-10 00:00:00+00:00 | CUST_mei_lin_66CBFE0F  | 96.2                   | 92.4                  

In [17]:
# Weekly trend for one customer
# ✏️ change customer_id to any of yours
CUSTOMER = "CUST_amara_patel_03630F04"

run_query(ts, f"""
    SELECT week, avg_heart_rate, avg_spo2,
           total_steps, avg_hrv, avg_resp_rate
    FROM wearable_weekly
    WHERE customer_id = '{CUSTOMER}'
    ORDER BY week DESC
""", f"Weekly Trend — {CUSTOMER}")


📊 Weekly Trend — CUST_amara_patel_03630F04
  week                   | avg_heart_rate         | avg_spo2               | total_steps            | avg_hrv                | avg_resp_rate         
  --------------------------------------------------------------------------------
  2026-05-04 00:00:00+00:00 | 87.9                   | 96.0                   | 28734                  | 46.79                  | 15.85                 
  2026-04-27 00:00:00+00:00 | 87.9                   | 96.0                   | 27461                  | 51.57                  | 15.99                 
  2026-04-20 00:00:00+00:00 | 87.5                   | 96.1                   | 28147                  | 49.86                  | 15.99                 
  2026-04-13 00:00:00+00:00 | 88.1                   | 96.1                   | 28963                  | 48.67                  | 16.03                 
  2026-04-06 00:00:00+00:00 | 87.8                   | 96.3                   | 8816                   | 47.19 

In [31]:
def check_raw_data(ts):
    with ts.cursor() as cur:
        cur.execute("""
            SELECT time, customer_id, heart_rate, spo2_pct, 
                   steps, hrv_ms, respiratory_rate
            FROM wearable_readings
            LIMIT 5
        """)
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
    print(" | ".join(f"{c:<18}" for c in cols))
    print("-" * 90)
    for row in rows:
        print(" | ".join(f"{str(v):<18}" for v in row))

check_raw_data(ts)

time               | customer_id        | heart_rate         | spo2_pct           | steps              | hrv_ms             | respiratory_rate  
------------------------------------------------------------------------------------------
2026-04-11 00:05:29+00:00 | CUST_amara_patel_03630F04 | None               | 94                 | 47                 | 52.0               | 12.0              
2026-04-11 01:45:29+00:00 | CUST_amara_patel_03630F04 | None               | 96                 | 229                | 71.0               | 12.0              
2026-04-11 02:44:29+00:00 | CUST_amara_patel_03630F04 | None               | 95                 | 110                | 41.0               | 13.0              
2026-04-11 03:24:29+00:00 | CUST_amara_patel_03630F04 | None               | 97                 | 309                | 22.0               | 19.0              
2026-04-11 04:59:29+00:00 | CUST_amara_patel_03630F04 | None               | 96                 | 321                | 75.0     

In [19]:
# High heart rate alert — readings above 100 bpm
run_query(ts, """
    SELECT time, customer_id, heart_rate, activity
    FROM wearable_readings
    WHERE heart_rate > 100
    ORDER BY heart_rate DESC
    LIMIT 10
""", "⚠️  High Heart Rate Alerts (> 100 bpm)")


📊 ⚠️  High Heart Rate Alerts (> 100 bpm)
  time                   | customer_id            | heart_rate             | activity              
  --------------------------------------------------------------------------------
  2026-04-25 19:27:29+00:00 | CUST_mei_lin_66CBFE0F  | 115                    | walking               
  2026-04-25 06:59:29+00:00 | CUST_mei_lin_66CBFE0F  | 115                    | light_activity        
  2026-05-07 08:03:29+00:00 | CUST_mei_lin_66CBFE0F  | 115                    | light_activity        
  2026-04-14 20:41:29+00:00 | CUST_mei_lin_66CBFE0F  | 115                    | light_activity        
  2026-04-26 17:20:29+00:00 | CUST_mei_lin_66CBFE0F  | 115                    | walking               
  2026-05-09 10:55:29+00:00 | CUST_mei_lin_66CBFE0F  | 114                    | resting               
  2026-04-30 03:41:29+00:00 | CUST_mei_lin_66CBFE0F  | 114                    | sleep                 
  2026-04-15 01:28:29+00:00 | CUST_mei_lin_66CBFE0F  |

In [20]:
# Monthly summary — all customers
run_query(ts, """
    SELECT month, customer_id,
           avg_heart_rate, avg_spo2,
           total_steps, avg_hrv
    FROM wearable_monthly
    ORDER BY month DESC, customer_id
""", "Monthly Summary — All Customers")


📊 Monthly Summary — All Customers
  month                  | customer_id            | avg_heart_rate         | avg_spo2               | total_steps            | avg_hrv               
  --------------------------------------------------------------------------------
  2026-04-16 00:00:00+00:00 | CUST_amara_patel_03630F04 | 87.8                   | 96.0                   | 101121                 | 49.16                 
  2026-04-16 00:00:00+00:00 | CUST_carlos_rivera_5A81755B | 69.6                   | 98.0                   | 152822                 | 50.27                 
  2026-04-16 00:00:00+00:00 | CUST_fatima_alsayed_8594AA83 | 72.8                   | 96.5                   | 49570                  | 47.36                 
  2026-04-16 00:00:00+00:00 | CUST_john_whitfield_C4987FD5 | 82.8                   | 95.0                   | 84174                  | 49.98                 
  2026-04-16 00:00:00+00:00 | CUST_mei_lin_66CBFE0F  | 97.1                   | 92.6                

---
#### 11.  Auto-Sync on a Schedule (Optional)

In [21]:
!pip install apscheduler --quiet

^C



[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
from datetime import datetime, timezone

def scheduled_sync():
    print(f"\n⏰ Auto-sync at {datetime.now(timezone.utc).strftime('%H:%M UTC')}")
    sync_wearables(get_minio(), get_ts())

scheduler = BackgroundScheduler()
scheduler.add_job(scheduled_sync, "interval", minutes=30)  # ✏️ adjust interval
scheduler.start()
print("⏰ Scheduler running — syncing every 30 min")
print("   Run scheduler.shutdown() to stop")

In [ ]:
# ⛔ Stop the scheduler
scheduler.shutdown()
print("Scheduler stopped.")

In [ ]:
# def drop_all(ts):
#     # close any open transaction first
#     ts.rollback()
#     ts.autocommit = True

#     with ts.cursor() as cur:
#         for view in ["wearable_hourly", "wearable_daily", "wearable_weekly", "wearable_monthly"]:
#             cur.execute(f"DROP MATERIALIZED VIEW IF EXISTS {view} CASCADE;")
#             print(f"  🗑️  Dropped view → {view}")

#         cur.execute("DROP TABLE IF EXISTS wearable_readings CASCADE;")
#         print("  🗑️  Dropped table → wearable_readings")

#     ts.autocommit = False
#     print("\n✅ All clear — now re-run from Cell 2 downward")

# drop_all(ts)

  🗑️  Dropped view → wearable_hourly
  🗑️  Dropped view → wearable_daily
  🗑️  Dropped view → wearable_weekly
  🗑️  Dropped view → wearable_monthly
  🗑️  Dropped table → wearable_readings

✅ All clear — now re-run from Cell 2 downward
